# Prostate MRI AI on CARPL: end-to-end proof

**For:** Dr Anh (radiologist and principal investigator) and the CARPL integration team.

This notebook runs real MRI studies through the exact containers that will be deployed on the SGH RAPIER host. It shows four things:

1. **It works end to end.** A study is pushed the way CARPL pushes it (DICOM C-STORE). The AI runs automatically, and the results come back to "CARPL" as DICOM.
2. **CARPL gets the right results.** The RTSTRUCT contours, drawn on the original T2 images, and the DICOM SR findings: PI-RADS, AI probability, lesion and prostate measurements.
3. **Nothing about the AI changed.** For every study, the CARPL output is compared with a standard run of the same AI on the original folder, the way HIVE runs it. They must be identical.
4. **It is safe.** A repeated study does not produce duplicate results. An incomplete study produces no result until the missing images arrive.

**How to read it:** every check ends in a **PASS / FAIL** table. The final table summarises everything.

**Data:** ProstateX-0004 (public TCIA dataset) and de-identified SGH research studies. The executed report contains images, so it is kept out of git (`carpl/proof_output/`).

**Re-running:** you need Docker with an NVIDIA GPU, and the images built by `python scripts/build_carpl_images.py --version <VERSION> --build-base`. Then run all cells, which takes about 20-30 minutes. Each run writes to a new folder and never touches existing data.

## How the pieces fit

```
CARPL ──C-STORE──▶ prostate-dicom-listener ──▶ prostate-redis ──▶ prostate-ai-worker (GPU)
                     waits for the whole study    job queue          runs the unchanged AI pipeline
CARPL ◀──C-STORE── prostate-dicom-sender ◀────────────────────────── only results that pass the audit
                     RTSTRUCT + SR
```

| Container | Role |
|---|---|
| `prostate-dicom-listener` | Receives the MRI from CARPL and waits until the study is complete. |
| `prostate-redis` | The to-do list between the containers; survives restarts. |
| `prostate-ai-worker` | Runs the **same** AI image used for HIVE, then audits the result. |
| `prostate-dicom-sender` | Sends the RTSTRUCT and SR back to CARPL, retrying if CARPL is busy. |

In this notebook a fifth container, `prostate-carpl-receiver`, stands in for CARPL's result receiver so we can inspect exactly what CARPL would receive.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "carpl" / "tools" / "proof_helpers.py").is_file())
sys.path.insert(0, str(REPO / "carpl" / "tools"))
import proof_helpers as ph

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

VERSION = "1.1.0"          # image tag built by scripts/build_carpl_images.py
TEST_DATA = REPO / "test-data"

# Studies shown with images: one public case, four SGH cases with a clear gland
# and lesion(s), and one SGH case with no lesion.
CASES = {
    "ProstateX-0004 (public)": TEST_DATA / "ProstateX-0004",
    "SGH A - 1 lesion": TEST_DATA / "20230316_649d127395cd81ca4195",
    "SGH B - 1 lesion": TEST_DATA / "20230222_2f4df4a94ccd37425b48",
    "SGH C - 1 lesion": TEST_DATA / "20230224_57f45ff07453250f8b8d",
    "SGH D - 2 lesions": TEST_DATA / "20230223_548530bb5b7523059fb6",
    "SGH E - no lesion": TEST_DATA / "20230223_bc28d473c48987fdac9c",
}
# Used only for the safety checks in step 8.
PARTIAL_CASE = TEST_DATA / "sgh_data_patient_2"

RUN = ph.new_run_folder()
DATA = RUN / "data"
print("This run writes to:", RUN.relative_to(REPO))

## Step 1: Check the computer is ready

In [ ]:
prereq = ph.check_prerequisites(VERSION)
display(prereq)
assert (prereq["result"] == "PASS").all(), "Fix the failed prerequisite before continuing"

## Step 2: Start the CARPL containers

This is the same `docker compose` set-up as production, except that the "CARPL" receiver is local. For speed, the study-complete wait is 20 s here; production uses 60 s.

In [ ]:
display(ph.start_stack(VERSION, DATA))
print("Worker GPU check:", ph.worker_preflight_line())

## Step 3: CARPL sends the MRI studies

Each study is sent exactly as CARPL sends it: a DICOM C-ECHO, then a C-STORE of every image in the study. The whole study is sent. The AI itself picks the T2, ADC and high-b series, as it does on HIVE.

In [ ]:
pushes = pd.DataFrame([ph.push_study(path, VERSION) | {"study": name} for name, path in CASES.items()])
display(pushes)
assert pushes["c_echo"].eq("0x0000").all() and pushes["c_store"].str.fullmatch(r"0x0000=\d+").all()

## Step 4: The AI runs and the results go back to CARPL

The listener waits until a study is complete, the worker runs the AI on the GPU, and the sender returns the results. Times are UTC.

In [ ]:
ph.wait_for_outcomes(DATA, count=len(CASES))
timeline = ph.study_timeline(DATA, CASES)
display(timeline)

## Step 5: What CARPL received

For every study CARPL receives exactly **two** DICOM objects, attached to the original study:

* **RT Structure Set**: prostate and lesion contours, the UroFusion-accepted format.
* **Comprehensive SR (TID 1500)**: AI findings and measurements, CARPL's preferred format (guide section 6.2).

In [ ]:
received = ph.received_objects(DATA, CASES)
display(received)

## Step 6: The results, as a radiologist would see them

For each study:

* The **contours CARPL received**, drawn on the **original T2 images**. Each panel is the T2 slice where that lesion is largest.
* The **SR findings**: one row per lesion. The lesion numbers match the RTSTRUCT names ("Lesion N - AI PI-RADS X").
* The **prostate measurements** in the same SR.

*Probability* is the peak value of the AI's 5-fold ensemble lesion probability inside that lesion. The AI only outlines a lesion where this is at least 63%. All results are AI-generated and marked **UNVERIFIED** pending radiologist review.

In [ ]:
import matplotlib.pyplot as plt

for name, path in CASES.items():
    display(Markdown(f"### {name} (`{path.name}`)"))
    rtstruct = ph.received_file(DATA, path, "RTSTRUCT")
    sr = ph.received_file(DATA, path, "SR")
    display(Markdown("**RTSTRUCT received by CARPL**"))
    display(ph.rtstruct_table(rtstruct))
    prostate, lesions, facts = ph.sr_findings(sr)
    display(Markdown("**SR: AI lesion findings**"))
    display(lesions)
    display(Markdown("**SR: prostate measurements**"))
    display(prostate.set_index("measurement").T)
    fig = ph.plot_contours(rtstruct, path, f"{name}: contours as received by CARPL")
    if fig is not None:
        plt.show()

## Step 7: The SR meets CARPL's DICOM SR requirements

This step checks every SR CARPL received against the CARPL DICOM Push Integration Guide, section 6.2 "Method 2: DICOM Structured Report (SR) - TID 1500".

In [ ]:
compliance = []
for name, path in CASES.items():
    table = ph.sr_compliance(ph.received_file(DATA, path, "SR"), path)
    table.insert(0, "study", name)
    compliance.append(table)
compliance = pd.concat(compliance, ignore_index=True)
display(compliance.pivot(index="CARPL requirement (guide 6.2)", columns="study", values="result"))
display(compliance[compliance["study"] == next(iter(CASES))][["CARPL requirement (guide 6.2)", "evidence"]])

## Step 8: Identical to the standard pipeline

For each study, the **unchanged inference image** is run directly on the **original study folder**, exactly as HIVE runs it. Its output is then compared with the CARPL output:

* **every NIfTI:** organ, cleaned organ, lesion mask, probability maps, T2/ADC/high-b, voxel by voxel;
* **`lesions.txt`:** PI-RADS, volumes and the cleanup audit, byte for byte;
* **the RTSTRUCT and SR content:** contours point by point, ignoring only the new UIDs and creation times each run gets;
* **the same T2, ADC and high-b series selected**, plus the existing RTSTRUCT round-trip validator.

One expected note: the SR lists its referenced T2 images in file-name order. Original files are named like `1-01.dcm`, while CARPL files are named by their DICOM UID, so the *order* of that list can differ. The *content* is identical, and this is reported as a note, not a failure.

In [ ]:
parity_rows = []
for name, path in CASES.items():
    direct = RUN / "direct" / path.name
    run_info = ph.direct_run(path, direct, VERSION)
    result = ph.parity(direct, ph.delivered_job_output(DATA, path), VERSION)
    parity_rows.append({"study": name, "direct run exit": run_info["exit_code"], "seconds": run_info["seconds"]}
                       | {k: v for k, v in result.items() if k != "raw"})
parity = pd.DataFrame(parity_rows)
display(parity)

## Step 9: Safety checks

1. **CARPL sends the same study again**, e.g. a retry. It is recognised as a duplicate, and no second result is created.
2. **An incomplete study arrives**: only the T2 series of a study. The AI cannot find ADC or high-b, so **nothing is sent** to CARPL.
3. **The missing images arrive later.** The study is re-assembled and processed in full, and the result is sent once.

In [ ]:
partial = {"SGH partial study": PARTIAL_CASE}
before = len([e for e in ph.events(DATA) if e["event"] in ph.TERMINAL_EVENTS])

resend = ph.push_study(CASES["ProstateX-0004 (public)"], VERSION)
t2_only = ph.push_study(PARTIAL_CASE, VERSION, include="^AX_t2_tse$")
ph.wait_for_outcomes(DATA, count=before + 2)
rest = ph.push_study(PARTIAL_CASE, VERSION, exclude="^AX_t2_tse$")
ph.wait_for_outcomes(DATA, count=before + 3)

display(pd.DataFrame([resend | {"step": "1. resend ProstateX-0004"},
                      t2_only | {"step": "2. send T2 series only"},
                      rest | {"step": "3. send the remaining series"}]))
safety_timeline = ph.study_timeline(DATA, {"ProstateX-0004 (public)": CASES["ProstateX-0004 (public)"]} | partial)
display(safety_timeline)
display(ph.received_objects(DATA, {"ProstateX-0004 (public)": CASES["ProstateX-0004 (public)"]} | partial))

## Summary

In [ ]:
px = safety_timeline[safety_timeline["study"] == "ProstateX-0004 (public)"]
part = safety_timeline[safety_timeline["study"] == "SGH partial study"]
received_all = ph.received_objects(DATA, {"ProstateX-0004 (public)": CASES["ProstateX-0004 (public)"]})
summary = pd.DataFrame([
    ("Containers start; GPU and models verified", (prereq["result"] == "PASS").all()),
    ("All studies accepted by DICOM C-STORE", pushes["c_store"].str.fullmatch(r"0x0000=\d+").all()),
    ("Every study processed and results delivered to CARPL",
     (timeline["final_state"] == "delivered").sum() == len(CASES)),
    ("CARPL received exactly RTSTRUCT + SR per study, on the right study",
     len(received) == 2 * len(CASES) and received["same_StudyInstanceUID"].all()),
    ("SR meets CARPL guide 6.2 (TID 1500 SR)", (compliance["result"] == "PASS").all()),
    ("CARPL output identical to the standard pipeline run", (parity["verdict"] == "PASS").all()),
    ("Repeated study: no duplicate result", (px["final_state"] == "duplicate_skipped").any() and len(received_all) == 2),
    ("Incomplete study: nothing sent until complete",
     (part["final_state"] == "failed").any() and (part["final_state"] == "delivered").sum() == 1),
], columns=["check", "passed"])
summary["result"] = summary["passed"].map({True: "PASS", False: "FAIL"})
display(summary[["check", "result"]])
print("ALL CHECKS PASSED" if summary["passed"].all() else "SOME CHECKS FAILED: see above")

## Step 10: Stop the containers

The run folder stays for inspection. It holds every job's `pipeline.log`, `job.json`, `state.json`, the audit CSV and what "CARPL" received.

In [ ]:
print(ph.stop_stack(VERSION, DATA))
print("Run folder:", RUN.relative_to(REPO))